In [7]:
import polars as pl
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier

# 1. Folders banao (Error nahi aayega)
os.makedirs("models", exist_ok=True)

# 2. Load dataset (Check karo ki data/student_por.csv file wahi hai)
df = pl.read_csv("data/student_por.csv") 

# 3. Preprocessing
df = df.with_columns((pl.col("G3") >= 10).cast(pl.Int64).alias("Pass"))

# Label Encoding
le = LabelEncoder()
df = df.with_columns(pl.Series("sex", le.fit_transform(df["sex"].to_numpy())))

# Selection
features = ["studytime", "absences", "sex"]
X = df.select(features).to_numpy()
y = df["Pass"].to_numpy()

# Feature Selection
selector = SelectKBest(score_func=f_classif, k=min(3, X.shape[1]))
X = selector.fit_transform(X, y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save Scaler (VERY IMPORTANT)
joblib.dump(scaler, "models/scaler.pkl")

# Training
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Save Model
joblib.dump(model, "models/model.pkl")

print("Model aur Scaler save ho gaye!")

# Prediction Function
def predict_student(data_row):
    loaded_model = joblib.load("models/model.pkl")
    loaded_scaler = joblib.load("models/scaler.pkl")
    data_scaled = loaded_scaler.transform(data_row)
    prediction = loaded_model.predict(data_scaled)
    return "PASS" if prediction[0] == 1 else "FAIL"

# Test
print("Sample Prediction:", predict_student(X_test[0:1]))

FileNotFoundError: The system cannot find the path specified. (os error 3): data/student_por.csv

In [8]:
import polars as pl
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier

# Folder setup
os.makedirs("models", exist_ok=True)

# Smart File Loading
file_path = "data/student_por.csv" if os.path.exists("data/student_por.csv") else "student_por.csv"
df = pl.read_csv(file_path)

# Data Processing
df = df.with_columns((pl.col("G3") >= 10).cast(pl.Int64).alias("Pass"))

le = LabelEncoder()
df = df.with_columns(pl.Series("sex", le.fit_transform(df["sex"].to_numpy())))

features = ["studytime", "absences", "sex"]
X = df.select(features).to_numpy()
y = df["Pass"].to_numpy()

selector = SelectKBest(score_func=f_classif, k=3)
X = selector.fit_transform(X, y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, "models/scaler.pkl")

models = {
    "Logistic": LogisticRegression(),
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"{name} | Acc: {accuracy_score(y_test, preds):.2f}")

grid = GridSearchCV(RandomForestClassifier(), {"n_estimators": [50, 100], "max_depth": [3, 5]}, cv=2)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
joblib.dump(best_model, "models/model.pkl")

print(f"\nModel saved. Best Params: {grid.best_params_}")

def predict_student(data_row):
    model = joblib.load("models/model.pkl")
    scaler = joblib.load("models/scaler.pkl")
    return "PASS" if model.predict(scaler.transform(data_row))[0] == 1 else "FAIL"

print("Sample Prediction:", predict_student(X_test[0:1]))

FileNotFoundError: The system cannot find the file specified. (os error 2): student_por.csv